In [ ]:
# ==========================================================
# Maestría en Ciencia y Análisis de Datos
# Universidad Mayor de San Andrés
# ----------------------------------------------------------
#   Modelos lineales y modelos lineales generalizados
# ----------------------------------------------------------
#        Rolando Gonzales Martinez, Julio 2026
# ==========================================================
#    Modelos lineales multiecuacionales: VAR(p) y BVAR(p)
# ==========================================================

import matplotlib.pyplot as plt
import pandas as pd

# Importar el conjunto de datos desde GitHub
url = 'https://raw.githubusercontent.com/rogon666/UMSA/refs/heads/main/2026/MLMLG_2026/datos/gdp_sectors.csv'

# Cargar los datos en un DataFrame
data = pd.read_csv(url)

# Convertir la fecha
data["mes"] = pd.to_datetime(data["mes"], format="%B-%y")

# Ordenar cronológicamente
data = data.sort_values("mes").reset_index(drop=True)

# Establecer la fecha como índice
data = data.set_index("mes")

print(data.head())

In [ ]:
# ==========================================================
# 1. Preparar los datos
# ==========================================================


data.index = pd.to_datetime(data.index)
data = data.sort_index()

# Variables sectoriales
variables = [
    "agr", "fish", "min", "manf",
    "elec", "cons", "serv", "taxs"
]

# ==========================================================
# 2. Graficar las series en niveles
# ==========================================================

ax = data[variables].plot(
    figsize=(13, 7),
    linewidth=1.8
)

ax.set_title("Evolución mensual de la producción por sector")
ax.set_xlabel("Fecha")
ax.set_ylabel("Nivel de producción")
ax.grid(True, alpha=0.3)
ax.legend(
    title="Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

# ==========================================================
# 3. Calcular variaciones porcentuales interanuales
# ==========================================================

# Variación porcentual respecto al mismo mes del año anterior:
# 100 * (Y_t / Y_{t-12} - 1)

variacion_12m = data[variables].pct_change(periods=12) * 100

# Alternativa equivalente:
# variacion_12m = (data[variables] / data[variables].shift(12) - 1) * 100

# Eliminar los primeros 12 meses, que no tienen comparación interanual
variacion_12m = variacion_12m.dropna(how="all")

print("\nVariaciones porcentuales a 12 meses:")
print(variacion_12m.round(4).tail())

# ==========================================================
# 4. Graficar variaciones porcentuales a 12 meses
# ==========================================================

ax = variacion_12m.plot(
    figsize=(13, 7),
    linewidth=1.8
)

ax.axhline(
    y=0,
    linewidth=1,
    linestyle="--"
)

ax.set_title("Variación porcentual a 12 meses por sector")
ax.set_xlabel("Fecha")
ax.set_ylabel("Variación interanual (%)")
ax.grid(True, alpha=0.3)
ax.legend(
    title="Sector",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------
# Seleccion de rezagos p en el modelo VAR(p)
# ------------------------------------------
from statsmodels.tsa.api import VAR

modelo = VAR(variacion_12m)

seleccion_rezagos = modelo.select_order(maxlags=1)

print(seleccion_rezagos.summary())

In [ ]:
# ==========================================
# Número de parámetros en un VAR(p)
# ==========================================

# Eliminar observaciones con valores faltantes
datos_var = variacion_12m.dropna()

# Número de observaciones efectivas
T = datos_var.shape[0]

# Número de variables
K = datos_var.shape[1]

# Número de rezagos
p = 6

# Número de términos determinísticos
# d = 1 -> constante
# d = 0 -> sin constante
d = 1

# Parámetros por ecuación
parametros_ecuacion = K * p + d

# Parámetros totales del sistema
parametros_totales = K * parametros_ecuacion

print(f"Muestra (T): {T}")
print(f"Número de variables (K): {K}")
print(f"Número de rezagos (p): {p}")
print(f"Parámetros por ecuación: {parametros_ecuacion}")
print(f"Parámetros del sistema completo: {parametros_totales}")

In [ ]:
# --------------------------------------------------------
# modelo VAR Bayesiano (BVAR)
# --------------------------------------------------------
import pymc as pm

# Estandarización
Y0 = variacion_12m.dropna()
Y0 = (Y0 - Y0.mean()) / Y0.std()

# 2. Matrices del BVAR(1): Y_t = c + A Y_{t-1} + u_t
Y = Y0.iloc[1:].to_numpy()
X = Y0.shift(1).iloc[1:].to_numpy()

T, K = Y.shape

# 3. Estimación bayesiana
with pm.Model() as bvar:

    # Constantes
    c = pm.Normal("c", mu=0, sigma=1, shape=K)

    # Prior de contracción: coeficientes cercanos a cero
    A = pm.Normal("A", mu=0,
                  sigma=0.30, # valor mas alto, mayor contraccion
                  shape=(K, K))

    # Desviación estándar de cada ecuación
    sigma = pm.HalfNormal("sigma", sigma=1, shape=K)

    # Media condicional
    mu = c + X @ A.T

    # Verosimilitud
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=Y)

    resultados = pm.sample(
        draws=2000,
        tune=1000,
        chains=4,
        target_accept=0.90,
        random_seed=666
    )

# 4. Matriz posterior media de coeficientes
A_media = resultados.posterior["A"].mean(("chain", "draw")).values

A_media = pd.DataFrame(
    A_media,
    index=variables,
    columns=[f"L1_{v}" for v in variables]
)

print(A_media.round(3))